In [1]:
from openai import OpenAI
from dotenv import load_dotenv
import os 

key_path = 'keys.env'
load_dotenv(dotenv_path=key_path) 
api_key = os.getenv("API_KEY_2")
client = OpenAI(api_key=api_key)

def ask_gpt(prompt):
    completion = client.chat.completions.create(
        model="gpt-5",
        messages=[
            {"role": "assistant", "content": "You are an expert in academic writing and citation analysis."},
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return completion.choices[0].message.content

In [2]:
format_def = """Issues in citation formatting such as a missing bracket and using the wrong style of citing.
    a) Due to preprocessing errors of the source dataset, some words contain hyphens that do not require it, and some are missing hyphens where it is required. Please ignore these types of formatting issues.
    b) Highlight the word/citation in which the formatting issue occurs in and not only the issue within the word/citation.
    c) Formatting issues appear as either citations or parts of a citation.
    Examples of formatting issues include:
        i) Narrative citation missing year: “Vatswani et al.” -> should be “Vatswani et al. (2020)”
        ii) Wrong citation style: “In (Vatswani et al., 2019)” -> should be “in Vatswani et al. (2019)”
        iii) Wrong use of footnotes: "Vastwani et al. 1" -> should include the year or be reformatted as a proper footnote."""

unsupp_def = """claim about prior work or statistics w/o citation or evidence. 
    a) The author should cite at every first mention of a study, paper, shared task, competition or dataset.
    b) Specific information to a niche topic, despite sounding like it should be known in that topic of study, should be cited.
    c) If a claim is made and is obvious to be a natural deduction from previous statements through common sense (i.e not requiring expert knowledge), then this claim does not fall under ‘Unsupported claim’. For example:
        i) “However, creating a large and suitable set of questions for supporting narrative comprehension is both time-consuming and cognitively demanding.” -> it is obvious that creating a dataset is time consuming and mentally demanding.
    d) Any mention of “recent works” should be backed up with citations to the works.
    e) Unsupported claim issues appear as segments, phrases, sub-sentences or full sentences.
    Examples of unsupported claims include:
        i) Missing citations for mentions of 'recent works': “and there are many recent works that explore this topic”,
        ii) Mention of a previous work and claim without citation: “..., while in a previous study, the authors claim …”,
        iii) Mentioning of a specific setup of a task without citation to the work: ".. BERT was used in an AES task trained on essays .." """

lacksynth_def = """occurs when either:
    a) The author describes or cites papers without connecting them to their own work/argument 
    b) Or only follows up the summary of previous works with their own contribution without explicitly highlighting the gap their work intends to research.
    c) It does not articulate the author's perspective or motivation.
    d) A lack of argument/opinion in the first paragraph is permissible as it serves to be the foundation of the author's argument 
    e) Lacks synthesis issues appear either as single sentences or multiple sentences.
    Examples of lack of synthesis include:
        i) No elaboration of own contribution/argument:"Following early neural approaches to question answering, many subsequent studies adopt a pipeline architecture consisting of retrieval and comprehension components. The retrieval component focuses on identifying relevant documents or passages from a large corpus, while the comprehension component extracts an answer span from the retrieved text. Initial models relied on recurrent neural networks with attention mechanisms to encode questions and contexts (Seo et al., 2017; Wang et al., 2017)."
        ii)  No explanation of the cited works and relation to their own work: “Recently, several studies have explored the use of prompting techniques with pre-trained language models to influence model outputs or access latent knowledge (Brown et al., 2020; Gao et al., 2021; Liu et al., 2021; Wei et al., 2022).” """

coherence_def = """connection between cited works is abrupt, lacking relation to each other. It is unclear how one mentioned work is relevant to a prior mentioned work. 
    a) Sentences are not transitioned from one to another.
    b) The relationship between sentences describing papers is implied but not explicitly stated.
    c) Coherence issues appear only as multiple sentences.
    Examples of coherence issues include:
        i) Relation between mentioned works is not explicit: “Smith (2020) identified a relationship between personal belief systems and ethical decision-making frameworks. Moral foundation theory proposes several core dimensions of moral reasoning, including harm, fairness, and authority (Jones, 2015). Audience adaptation has been explored in computational argumentation. Lee et al. (2019) applied moral categories to argument generation tasks. Human annotators often disagree when labeling moral dimensions in text (Nguyen et al., 2018).”
        ii) Lack of transitions between sentences: “Recent studies have explored various techniques for enhancing model performance. Smith et al. (2020) introduced a novel architecture that significantly improves accuracy on benchmark datasets. Additionally, Johnson and Lee (2019) proposed a data augmentation method that increases training data diversity.” 
        iii) No explanation of the cited works and relation to their own work: “Recently, several studies have explored the use of prompting techniques with pre-trained language models to influence model outputs or access latent knowledge (Brown et al., 2020; Gao et al., 2021; Liu et al., 2021; Wei et al., 2022).” """

category_definitions = {'Unsupported claim': unsupp_def,
                'Format': format_def,
                'Coherence': coherence_def,
                'Lacks synthesis': lacksynth_def}

In [3]:
category_constraints = {
    "Coherence": """- The document should mention at least 3 distinct prior works in sequence.
- The new span should create an implicit topic shift or weak connection WITHOUT explicit transition markers.
- Avoid discourse connectors such as "however", "in contrast", "similarly", "therefore", "moreover" inside the new span.""",
    "Lacks synthesis": """- The new span should read like a literature list: mention at least 4 works/datasets/methods.
- The span must NOT include author stance or synthesis (avoid: "we argue", "we propose", "this suggests", "overall").
- The paragraph should feel complete but still lacks the author's own connecting argument.""",
    "Unsupported claim": """- The new span must include a generalizing claim about prior work without citation/evidence.
- The claim should reference a specific method/dataset/trend (e.g., "two-stage retriever-reader") but NOT cite who did it.
- Use realistic academic phrasing such as "recent studies", "prior work", "widely adopted approaches".""",
    "Format": """- The new span must be ONLY the citation text (not the surrounding sentence).
- Introduce realistic formatting mistakes: missing bracket, wrong style (narrative vs parenthetical), truncated year, etc.
- Keep it plausible under the paper's citation style.""",
}

In [4]:
### helper functions 
import json 

def parse_gpt_json(response):
    json_structure = json.loads(response)
    return json_structure

### Critic LLM response

In [5]:
def critic_turn(generator_response, category, span_1, span_2):
    constraint = category_constraints[category]
    definition = category_definitions[category]

    prompt = f"""
Your task is to judge an academic document that contains spans for the issue "{category}", and determine whether the style and structure of both the span AND document are similar to the example document provided.
More specifically, the DOCUMENT should follow these constraints:

Category specific constraint:
{constraint}

And the SPANS must follow this definition for the category "{category}":
{definition}

Here are some document and span examples of "{category}":

Span: "{span_1['text']}"
Document:
{span_1['full_text']}

Span: "{span_2['text']}"
Document:
{span_2['full_text']}

End of examples.

Here is the document with spans you should evaluate:
Document:
{generator_response['document']}

Spans:
{generator_response['spans']}

Your final decision of whether the spans and document follow the definition and constraints should be:
EITHER 1 (yes, the document or span is suitable) OR 0 (no, the document or span is not suitable) for each span and document.
You may reason and provide it in the 'reason' key. Return ONLY a JSON array in this form:
{{
  "spans": [0 or 1, ..., ...],
  "document": 0 or 1,
  "reason": "..."
}}
    """

    critic_response = ask_gpt(prompt)
    critic_json = parse_gpt_json(critic_response)
    if len(critic_json['spans']) != len(generator_response['spans']):
          critic_response = ask_gpt(prompt)
          critic_json = parse_gpt_json(critic_response)
          
    return critic_json
    

In [9]:
synth_path = "samples/synthetic_samples_4.json"
categories = ["Unsupported claim", "Lacks synthesis", "Format", "Coherence"]

with open(synth_path, 'r') as f:
    synth_data = json.load(f)

total = {"Unsupported claim": [], "Lacks synthesis": [], "Format": [], "Coherence": []}
for category in categories:
    docs_to_spans = {}

    for sample in synth_data.get(category, []):
        doc = sample["document"]
        span_entry = {
            "span": sample["span"],
            "start": sample["start"],
            "end": sample["end"],
        }

        if doc not in docs_to_spans:
            docs_to_spans[doc] = []
        docs_to_spans[doc].append(span_entry)

    for doc, spans in docs_to_spans.items():
        total[category].append({
            "document": doc,
            "spans": spans,
        })


In [7]:
# grouping real samples by categories
with open("samples/for_synth_sampling.json", 'r') as f:
    real_samples = json.load(f)

total_real = {"Unsupported claim": [], "Lacks synthesis": [], "Format": [], "Coherence": []}
for category in categories:
    for paper in real_samples:
        for samples in real_samples[paper]:
            if samples['label'] == category:
                total_real[category].append(samples)

In [18]:
synth_path = "samples/synthetic_samples_4.json"
categories = ["Unsupported claim", "Lacks synthesis", "Format", "Coherence"]

with open(synth_path, 'r') as f:
    total = json.load(f)

In [23]:
import random 
from tqdm import tqdm

random.seed(42)

final_dataset = {"Unsupported claim": [], "Lacks synthesis": [], "Format": [], "Coherence": []}
final_critiques = {"Unsupported claim": [], "Lacks synthesis": [], "Format": [], "Coherence": []}

for category in categories:
    # examples for critic gpt to use as reference
    example_1 = random.choice(total_real[category])
    example_2 = random.choice(total_real[category])
    while example_1['text'] == example_2['text']:
        example_2 = random.choice(total_real[category])

    for samples in tqdm(total[category], desc=f"Evaluating {category} samples: "):
        critic_json = critic_turn(samples, category, example_1, example_2)
        final_sample = {"document": samples['document'], 'spans': []}

        for i in range(len(critic_json['spans'])):
            if critic_json['spans'][i] == 1:
                final_sample['spans'].append(samples['spans'][i])

        final_dataset[category].append(final_sample)
        final_critiques[category].append(critic_json)

    filename = category.replace(" ", "_").lower()
    with open(f"samples/{filename}_critic_evalauted.json", 'w') as f:
        json.dump(final_dataset[category], f, indent=2)
    with open(f"samples/critiques/{filename}_critiques.json", 'w') as f:
        json.dump(final_critiques[category], f, indent=2)

Evaluating Coherence samples: 100%|██████████| 300/300 [2:09:51<00:00, 25.97s/it]  


In [21]:
import json
categories = ['Unsupported claim', 'Lacks synthesis', 'Format', 'Coherence']

with open("samples/total_synth_data.json", 'r') as f:
    total = json.load(f)

final_set = {'Unsupported claim': [], 'Lacks synthesis': [], 'Format': [], 'Coherence': []}
for category in categories:
    for samples in total[category]:
        if len(samples['spans']) != 0:
            final_set[category].append(samples)

total = final_set

In [22]:
categories = ['Unsupported claim', 'Lacks synthesis', 'Format', 'Coherence']

def extract_start_end(span, document, used_positions=None):
    if used_positions is None:
        used_positions = set()

    start = document.find(span)
    while start != -1:
        if start not in used_positions:
            end = start + len(span)
            return start, end
        start = document.find(span, start + 1)

    return -1, -1


for category in categories:
    for i in range(len(total[category])):
        document = total[category][i]['document']
        spans = total[category][i]['spans']
        span_entries = []
        used_positions = set()

        for span in spans:
            start, end = extract_start_end(span, document, used_positions)
            if start != -1:
                used_positions.add(start)

            span_entries.append({
                "span": span,
                "start": start,
                "end": end
            })

        total[category][i]['spans'] = span_entries

In [28]:
with open("samples/synthetic_samples_complete.json", 'w') as f:
    json.dump(total, f, indent=2)